# DELICHOICE — Notebook de Reproducción (CRUD · Valoración · Recomendación)

**Titulo Proyecto:** *JIRA*

**Grupo:** *ABP_3*  
**Autores:** *Juan Garrido Blanco Imanol Justel Míguez Rubén Bouzón Lago Aarón Blanco Rodríguez*   
**Fecha:** *09/01/2026*  

---

Este notebook reúne, en un único documento, lo necesario para **reproducir el funcionamiento de la aplicación**:

1. **CRUD de datos** (Sprint 2)  
2. **Sistema de valoración** (relacionado con el registro y tratamiento de puntuaciones)  
3. **Sistema de recomendación** (Sprint 3)

> **Objetivo:** que cualquier persona pueda ejecutar las celdas en orden y entender **qué hace cada módulo, cómo se usa y qué recursos necesita**.


## Requisitos y ejecución

### Requisitos software
- Python 3.9+ (recomendado)
- Librerías: `pandas`, `numpy`, `scikit-learn`

### Cómo ejecutar
1. Abrir este `.ipynb` en Jupyter / VSCode / Colab.
2. Ejecutar las celdas **de arriba a abajo**.
3. Si usas un dataset externo (CSV), colócalo en la misma carpeta del notebook o ajusta la ruta indicada.

> Si alguna librería no está instalada, puedes usar:  
> `pip install pandas numpy scikit-learn`


---
# Parte 1 — CRUD de datos (Sprint 2)

A continuación se incluye el contenido del notebook de Sprint 2 (CRUD), con un **manual de uso** integrado.


## Cómo está organizado este notebook

- **Parte 1 (CRUD):** gestión de datos de restaurantes (crear, leer, actualizar, borrar).
- **Parte 3 (Recomendador):** recomendaciones basadas en similitud de texto (TF‑IDF + coseno).
- Entre ambas partes se incluye una explicación de la **estructura del CSV** del proyecto (dataset de 10.000 restaurantes).

> Consejo: ejecuta las celdas en orden. Si reinicias el kernel, vuelve a ejecutar desde el inicio.


## Manual de uso — CRUD (Sprint 2)

Esta sección demuestra las operaciones **CRUD** sobre el conjunto de restaurantes:

- **Create (Crear):** añadir un nuevo restaurante.
- **Read (Leer):** listar restaurantes o consultar uno en concreto.
- **Update (Actualizar):** modificar campos (nombre, tipo de cocina, localización, descripción…).
- **Delete (Borrar):** eliminar un restaurante.

### Cómo usarlo (pasos recomendados)
1. Ejecuta las celdas de **importaciones/configuración**.
2. Ejecuta la celda que **carga/inicializa** el almacenamiento de datos (p. ej. JSON/archivo local).
3. Prueba el CRUD en este orden:
   - Crear 1–2 registros de ejemplo.
   - Listarlos para verificar que se han guardado.
   - Actualizar algún campo y volver a listar.
   - Borrar un registro y comprobar que desaparece.

### Qué debes observar al ejecutar
- Qué estructura de datos se guarda (campos mínimos del restaurante).
- Qué función o celda corresponde a cada operación CRUD.
- Cómo se persisten los cambios (archivo local / estructura en memoria).


## Manual de uso — DELICHOICE (Notebook)
**Autores:** Juan Garrido Blanco, Imanol Justel Míguez, Rubén Bouzón Lago, Aarón Blanco Rodríguez  
**Centro:** Escola Superior de Enxeñería Informática (ESEI)  

---

## Índice
1. Introducción
2. Requisitos previos
3. Carga y almacenamiento de datos
4. Funciones CRUD (create/read/update/delete)
5. Dataset de ejemplo
6. Pruebas CRUD básicas
7. Validaciones y gestión de errores
8. Visualización (tabla)
9. Resultados esperados y conclusiones

## 1. Introducción
Este notebook sirve como **manual de uso** para demostrar las pruebas de **carga y tratamiento de datos** en DELICHOICE.
Implementa un CRUD de *items* (restaurantes) con:
- Almacenamiento en JSON local
- Validaciones mínimas
- Funciones `create`, `list/get`, `update`, `delete`
- Visualización en tabla

> Este contenido está pensado para acompañar el trabajo de DELICHOICE y mostrar un flujo reproducible en Google Colab.


## 2. Requisitos previos
- Entorno: **Google Colab**
- (Opcional) `pandas` para visualizar tablas (en Colab ya viene instalado).
- No se requiere base de datos externa; se usa un archivo **JSON** en `/content/items_store.json`.


In [ ]:
import json, os, uuid
from typing import Dict, Any, List

print("🔧 Configurando entorno y rutas...")

# Ruta donde se guardarán los datos
DATA_PATH = "/content/items_store.json"

def _ensure_store(path: str = DATA_PATH) -> None:
    """Crea un archivo JSON vacío si no existe."""
    if not os.path.exists(path):
        print("📁 Archivo JSON no encontrado. Creando nuevo archivo vacío...")
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"items": []}, f, ensure_ascii=False, indent=2)
    else:
        print("✅ Archivo JSON encontrado.")

def _load(path: str = DATA_PATH) -> Dict[str, Any]:
    """Carga el contenido del archivo JSON."""
    _ensure_store(path)
    with open(path, "r", encoding="utf-8") as f:
        print("📖 Cargando datos desde el archivo JSON...")
        return json.load(f)

def _save(payload: Dict[str, Any], path: str = DATA_PATH) -> None:
    """Guarda el contenido actualizado en el archivo JSON."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print("💾 Datos guardados correctamente en", path)



🔧 Configurando entorno y rutas...


In [ ]:
def validate_item(payload: Dict[str, Any]) -> List[str]:
    """Validación mínima para un restaurante."""
    print("🧩 Validando datos del restaurante...")
    errors = []
    if "name" not in payload or not str(payload["name"]).strip():
        errors.append("`name` es requerido y no puede estar vacío.")
    if "category" not in payload or not str(payload["category"]).strip():
        errors.append("`category` es requerido.")
    if "price" not in payload or not isinstance(payload["price"], (int, float)) or payload["price"] < 0:
        errors.append("`price` es requerido y debe ser >= 0.")
    if errors:
        print("⚠️ Se encontraron errores de validación:", errors)
    else:
        print("✅ Validación superada.")
    return errors


def create_item(item: Dict[str, Any], path: str = DATA_PATH) -> Dict[str, Any]:
    """Crea y persiste un restaurante (item)."""
    print("🆕 Creando nuevo restaurante...")
    errors = validate_item(item)
    if errors:
        raise ValueError("Errores de validación: " + "; ".join(errors))
    db = _load(path)
    new_item = {
        "id": item.get("id", str(uuid.uuid4())),
        "name": item["name"].strip(),
        "category": item["category"].strip(),
        "price": float(item["price"]),
        "location": item.get("location", "").strip(),
        "tags": list(item.get("tags", [])),
        "active": bool(item.get("active", True))
    }
    db["items"].append(new_item)
    _save(db, path)
    print(f"✅ Restaurante '{new_item['name']}' creado con ID {new_item['id']}")
    return new_item


def list_items(path: str = DATA_PATH) -> List[Dict[str, Any]]:
    """Lista todos los restaurantes."""
    print("📋 Listando todos los restaurantes almacenados...")
    return _load(path)["items"]


def get_item(item_id: str, path: str = DATA_PATH) -> Dict[str, Any] | None:
    """Obtiene un restaurante por ID."""
    print(f"🔍 Buscando restaurante con ID: {item_id}")
    for it in list_items(path):
        if it["id"] == item_id:
            print("✅ Restaurante encontrado:", it)
            return it
    print("❌ No se encontró un restaurante con ese ID.")
    return None


def update_item(item_id: str, patch: Dict[str, Any], path: str = DATA_PATH) -> Dict[str, Any]:
    """Actualiza un restaurante existente."""
    print(f"✏️ Actualizando restaurante con ID: {item_id}")
    db = _load(path)
    for i, it in enumerate(db["items"]):
        if it["id"] == item_id:
            updated = it.copy()
            updated.update(patch)
            errors = validate_item(updated)
            if errors:
                raise ValueError("Errores de validación: " + "; ".join(errors))
            db["items"][i] = updated
            _save(db, path)
            print("✅ Restaurante actualizado correctamente:", updated)
            return updated
    raise KeyError(f"❌ No se encontró el restaurante con id {item_id}.")


def delete_item(item_id: str, path: str = DATA_PATH) -> bool:
    """Elimina un restaurante por ID."""
    print(f"🗑️ Eliminando restaurante con ID: {item_id}")
    db = _load(path)
    before = len(db["items"])
    db["items"] = [it for it in db["items"] if it["id"] != item_id]
    _save(db, path)
    if len(db["items"]) < before:
        print("✅ Restaurante eliminado correctamente.")
        return True
    else:
        print("⚠️ No se encontró el restaurante a eliminar.")
        return False


In [ ]:
print("📦 Creando dataset de ejemplo...")

_ensure_store()

sample = {
    "items": [
        {"id": "r-sushi-001", "name": "Sushi Hana", "category": "japonés", "price": 15.9, "location": "Centro", "tags": ["sushi","cena"], "active": True},
        {"id": "r-pizza-002", "name": "La Trattoria", "category": "italiano", "price": 11.5, "location": "Norte", "tags": ["pizza","grupos"], "active": True},
        {"id": "r-ramen-003", "name": "Ramen Co", "category": "japonés", "price": 13.0, "location": "Sur", "tags": ["sopa","caliente"], "active": True},
    ]
}
_save(sample)
print("✅ Dataset inicial creado correctamente.")
print("📊 Contenido actual:", _load())



📦 Creando dataset de ejemplo...
✅ Archivo JSON encontrado.
💾 Datos guardados correctamente en /content/items_store.json
✅ Dataset inicial creado correctamente.
✅ Archivo JSON encontrado.
📖 Cargando datos desde el archivo JSON...
📊 Contenido actual: {'items': [{'id': 'r-sushi-001', 'name': 'Sushi Hana', 'category': 'japonés', 'price': 15.9, 'location': 'Centro', 'tags': ['sushi', 'cena'], 'active': True}, {'id': 'r-pizza-002', 'name': 'La Trattoria', 'category': 'italiano', 'price': 11.5, 'location': 'Norte', 'tags': ['pizza', 'grupos'], 'active': True}, {'id': 'r-ramen-003', 'name': 'Ramen Co', 'category': 'japonés', 'price': 13.0, 'location': 'Sur', 'tags': ['sopa', 'caliente'], 'active': True}]}


In [ ]:
print("\n🚀 Iniciando pruebas CRUD básicas...\n")

# CREATE
nuevo = create_item({
    "name": "Casa Pepe",
    "category": "mediterráneo",
    "price": 9.9,
    "location": "Centro",
    "tags": ["tapas","barato"],
    "active": True
})
print("🆕 Creado:", nuevo)

# READ
print("\n📘 Listando todos los restaurantes actuales:")
print(list_items())

# GET
print("\n🔎 Buscando restaurante específico:")
print(get_item("r-sushi-001"))

# UPDATE
print("\n✏️ Actualizando restaurante existente...")
actualizado = update_item("r-sushi-001", {"price": 16.4, "tags": ["sushi","cena","pop"]})
print("🆙 Actualizado:", actualizado)

# DELETE
print("\n🗑️ Eliminando restaurante...")
eliminado = delete_item("r-pizza-002")
print("✅ Eliminado:", eliminado)

print("\n📋 Estado final del JSON:")
print(list_items())




🚀 Iniciando pruebas CRUD básicas...

🆕 Creando nuevo restaurante...
🧩 Validando datos del restaurante...
✅ Validación superada.
✅ Archivo JSON encontrado.
📖 Cargando datos desde el archivo JSON...
💾 Datos guardados correctamente en /content/items_store.json
✅ Restaurante 'Casa Pepe' creado con ID 5850ac3e-92d7-4247-8400-d979277db712
🆕 Creado: {'id': '5850ac3e-92d7-4247-8400-d979277db712', 'name': 'Casa Pepe', 'category': 'mediterráneo', 'price': 9.9, 'location': 'Centro', 'tags': ['tapas', 'barato'], 'active': True}

📘 Listando todos los restaurantes actuales:
📋 Listando todos los restaurantes almacenados...
✅ Archivo JSON encontrado.
📖 Cargando datos desde el archivo JSON...
[{'id': 'r-sushi-001', 'name': 'Sushi Hana', 'category': 'japonés', 'price': 15.9, 'location': 'Centro', 'tags': ['sushi', 'cena'], 'active': True}, {'id': 'r-pizza-002', 'name': 'La Trattoria', 'category': 'italiano', 'price': 11.5, 'location': 'Norte', 'tags': ['pizza', 'grupos'], 'active': True}, {'id': 'r-ram

In [ ]:
print("\n🧱 Pruebas de validación (errores controlados)...")

def probar_creacion_invalida(payload):
    try:
        create_item(payload)
    except Exception as e:
        print("❌ Error esperado:", e)

# Falta 'name'
probar_creacion_invalida({"price": 10, "category": "mexicano"})

# Precio negativo
probar_creacion_invalida({"name": "Taquería X", "price": -2, "category": "mexicano"})

# Falta 'category'
probar_creacion_invalida({"name": "Cafetería Y", "price": 3.5})



🧱 Pruebas de validación (errores controlados)...
🆕 Creando nuevo restaurante...
🧩 Validando datos del restaurante...
⚠️ Se encontraron errores de validación: ['`name` es requerido y no puede estar vacío.']
❌ Error esperado: Errores de validación: `name` es requerido y no puede estar vacío.
🆕 Creando nuevo restaurante...
🧩 Validando datos del restaurante...
⚠️ Se encontraron errores de validación: ['`price` es requerido y debe ser >= 0.']
❌ Error esperado: Errores de validación: `price` es requerido y debe ser >= 0.
🆕 Creando nuevo restaurante...
🧩 Validando datos del restaurante...
⚠️ Se encontraron errores de validación: ['`category` es requerido.']
❌ Error esperado: Errores de validación: `category` es requerido.


In [ ]:
import pandas as pd

print("\n📈 Mostrando tabla de restaurantes actualizados...")
df = pd.DataFrame(list_items())
display(df)



📈 Mostrando tabla de restaurantes actualizados...
📋 Listando todos los restaurantes almacenados...
✅ Archivo JSON encontrado.
📖 Cargando datos desde el archivo JSON...


,id,name,category,price,location,tags,active
0,r-sushi-001,Sushi Hana,japonés,16.4,Centro,"[sushi, cena, pop]",True
1,r-ramen-003,Ramen Co,japonés,13.0,Sur,"[sopa, caliente]",True
2,5850ac3e-92d7-4247-8400-d979277db712,Casa Pepe,mediterráneo,9.9,Centro,"[tapas, barato]",True


## 9. Resultados esperados y conclusiones
- Tras ejecutar las celdas, el archivo **/content/items_store.json** refleja todas las operaciones CRUD.
- La tabla final muestra el estado actualizado de los restaurantes.
- Las **validaciones** evitan datos inconsistentes (precio negativo, campos obligatorios vacíos).
- Este notebook sirve como **manual de uso** reproducible de la herramienta de pruebas de DELICHOICE.

### Posibles problemas y soluciones
- **Archivo no aparece:** ejecutar la celda de “Carga y almacenamiento de datos” para crear el JSON.
- **Errores de validación:** revisa los campos obligatorios (`name`, `category`, `price >= 0`).
- **Sobrescritura de datos:** vuelve a ejecutar la celda “Dataset de ejemplo” para reponer el estado inicial.


## Estructura del dataset CSV (restaurantes_delichoice_10000_unique_contenido.csv)

Este proyecto utiliza un dataset con **10.000 restaurantes** (una fila por restaurante).  
El fichero contiene **7 columnas**:

| Columna | Tipo | Descripción |
|---|---|---|
| `id` | int | Identificador único del restaurante. |
| `nombre` | str | Nombre comercial del restaurante. |
| `tipo_cocina` | str | Tipo de cocina (p. ej. *japonesa*, *gallega*, *mexicana*). |
| `localizacion` | str | Ciudad o zona (en nuestro dataset: ciudades como Málaga, Madrid, Bilbao, etc.). |
| `descripcion` | str | Texto descriptivo del restaurante (ambiente, estilo, especialidad…). |
| `puntuacion_media` | float | Media de valoraciones (habitualmente en escala 0–5). |
| `num_valoraciones` | int | Número de valoraciones usadas para calcular la media. |

### ¿Para qué sirve cada campo en la aplicación?
- En el **CRUD** se usa para **crear/listar/editar** restaurantes (campos básicos: `nombre`, `tipo_cocina`, `localizacion`, `descripcion`).
- En la **valoración** (si se usa en la app) `puntuacion_media` y `num_valoraciones` permiten mostrar “popularidad/calidad”.
- En el **recomendador**, el corazón es el **texto** (principalmente `descripcion` + campos relacionados como `tipo_cocina`), que se transforma a vectores (TF‑IDF) para comparar similitudes entre restaurantes.

### Ejemplo de fila (primeras columnas)
A modo ilustrativo, una fila típica tiene esta forma:

- `id`: 1
- `nombre`: Fonda Do Sol
- `tipo_cocina`: japonesa
- `localizacion`: Málaga
- `puntuacion_media`: 4.06
- `num_valoraciones`: 330

> Nota: el recomendador necesita **una columna de texto**. En nuestro CSV la aporta `descripcion`.


---
# Parte 2 — Sistema de recomendación (Sprint 3)

A continuación se integra el notebook del recomendador (Sprint 3) con un **manual de uso** y una carga de datos más robusta.


## Manual de uso — Recomendador (Sprint 3)

Esta sección implementa un sistema de recomendación **basado en contenido** (*content-based*):

1. Se construye una representación textual del restaurante (normalmente con `descripcion`, `tipo_cocina`, etc.).
2. Se convierte el texto a vectores usando **TF‑IDF**.
3. Se calcula la similitud entre restaurantes mediante **similitud del coseno**.
4. Dado un restaurante (o una consulta), se devuelven los **más similares** como recomendaciones.

### Cómo usarlo (pasos recomendados)
1. Ejecuta la celda de **carga del CSV**.
2. Ejecuta las celdas de **preprocesado** (limpieza básica / creación del corpus).
3. Ejecuta la celda que **entrena/calcula** la matriz de similitud (TF‑IDF + coseno).
4. Llama a la función del notebook que obtiene recomendaciones, indicando:
   - un `id` o un `nombre` de restaurante (según esté implementado),
   - el número `top_n` de recomendaciones.

### Qué devuelve y cómo interpretarlo
- Un ranking de restaurantes similares (por texto).
- Cuanto mayor la similitud, más se parecen en descripción/tipo de cocina.
- Si aparece “ruido”, suele deberse a descripciones genéricas; se puede mejorar enriqueciendo el texto de entrada.


## DELICHOICE – Sistema de Recomendación Basado en Contenido
## Grupo JIRA

---

## 📌 Integrantes:
- Juan Garrido Blanco  
- Imanol Justel Míguez  
- Rubén Bouzón Lago  
- Aarón Blanco Rodríguez  

---

## 🎯 Goals

- Preprocesar el texto del campo `descripcion`.
- Basado en TF-IDF.
- Emplea **distintos métodos de similitud**:  
  - Coseno  
  - Euclidiana  
  - Manhattan

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import (
    cosine_similarity,
    euclidean_distances,
    manhattan_distances
)

pd.set_option("display.max_colwidth", 200)


## 2. Carga del dataset

Usaremos el archivo:

**restaurantes_delichoice_10000_unique_contenido.csv**

Debe estar en el mismo directorio que el notebook.


In [ ]:
# --- Carga de dataset para el recomendador ---
# Opción A (recomendada): usar el CSV grande del proyecto (si existe).
# Opción B: usar el CSV exportado desde el CRUD (Parte 2).

import os
import pandas as pd

possible_paths = [
    "restaurantes_delichoice_10000_unique_contenido.csv",  # CSV del proyecto (si lo tienes)
    "restaurantes_export_crud.csv",                        # generado por este notebook
]

csv_path = None
for p in possible_paths:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError(
        "No se encontró ningún CSV. Coloca el dataset en la carpeta del notebook "
        "o ejecuta la exportación del CRUD (Parte 2)."
    )

df = pd.read_csv(csv_path)
print("✅ Dataset cargado:", csv_path, "| filas:", len(df), "| columnas:", list(df.columns))

# Normalizamos columnas mínimas esperadas por el recomendador
required_cols = ["id", "nombre"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"El dataset debe incluir la columna obligatoria: {col}")

# Columna textual para TF-IDF
if "contenido" not in df.columns:
    # fallback razonable: nombre como contenido
    df["contenido"] = df["nombre"].astype(str)

# Señales de valoración (si no existen, se crean con 0)
if "puntuacion_media" not in df.columns:
    df["puntuacion_media"] = 0.0
if "num_valoraciones" not in df.columns:
    df["num_valoraciones"] = 0

df.head()


✅ Dataset cargado: restaurantes_delichoice_10000_unique_contenido.csv | filas: 10000 | columnas: ['id', 'nombre', 'tipo_cocina', 'localizacion', 'descripcion', 'puntuacion_media', 'num_valoraciones']


,id,nombre,tipo_cocina,localizacion,descripcion,puntuacion_media,num_valoraciones,contenido
0,1,Fonda Do Sol,japonesa,Málaga,"Especializado en carnes a la brasa de larga maduración, resulta ideal para eventos de empresa. Dispone de menú del día con varias alternativas. Restaurante de cocina mexicana situado en un entorno...",4.06,330,Fonda Do Sol
1,2,Burger Bella,gallega,Madrid,"La cocina es visible desde la sala principal. La mayoría de los ingredientes son de proximidad. Especializado en tacos mexicanos con salsas caseras, resulta ideal para comidas rápidas entre semana...",4.75,179,Burger Bella
2,3,Ramen Select,mexicana,Bilbao,La cocina es visible desde la sala principal. El equipo de cocina apuesta por sabores intensos. Restaurante de cocina italiana situado en un entorno animado y perfecto para grupos. Especializado e...,3.90,344,Ramen Select
3,4,Taberna Rica,japonesa,A Coruña,"Especializado en carnes a la brasa de larga maduración, resulta ideal para reuniones informales. Restaurante de cocina vegana situado en un entorno tranquilo y silencioso. La música de fondo crea ...",4.37,492,Taberna Rica
4,5,Burger Gourmet,vegana,Barcelona,Restaurante de cocina mexicana situado en un entorno elegante pero cercano. La presentación de los platos está muy cuidada. El servicio es cercano y cuida cada detalle. Especializado en ramen case...,4.11,96,Burger Gourmet


In [ ]:
df.info()
df.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                10000 non-null  int64  
 1   nombre            10000 non-null  object 
 2   tipo_cocina       10000 non-null  object 
 3   localizacion      10000 non-null  object 
 4   descripcion       10000 non-null  object 
 5   puntuacion_media  10000 non-null  float64
 6   num_valoraciones  10000 non-null  int64  
 7   contenido         10000 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 625.1+ KB


,id,nombre,tipo_cocina,localizacion,descripcion,puntuacion_media,num_valoraciones,contenido
5768,5769,Curry House,mexicana,Ourense,"Especializado en tacos mexicanos con salsas caseras, resulta ideal para comidas de negocio. El equipo de cocina apuesta por sabores intensos. Restaurante de cocina italiana situado en un entorno i...",3.97,106,Curry House
3342,3343,Veggie Rica,japonesa,Sevilla,"Especializado en hamburguesas gourmet con pan brioche, resulta ideal para reuniones informales. Incluye opciones sin gluten bajo petición. Restaurante de cocina mexicana situado en un entorno romá...",4.54,310,Veggie Rica
5347,5348,Osteria Do Sol,fusión,Ourense,"El equipo de cocina apuesta por sabores intensos. Restaurante de cocina italiana situado en un entorno elegante pero cercano. Especializado en platos de cuchara tradicionales, resulta ideal para c...",4.23,32,Osteria Do Sol
9532,9533,Asador Zen,japonesa,Santiago,"El espacio combina mesas altas y bajas. Especializado en ensaladas healthy con productos de temporada, resulta ideal para cenas en pareja. Restaurante de cocina japonesa situado en un entorno urba...",3.73,144,Asador Zen
1205,1206,Ramen Roma,italiana,Málaga,"Especializado en postres caseros inspirados en recetas locales, resulta ideal para brunch de fin de semana. El local admite reservas para grupos grandes. Restaurante de cocina china situado en un ...",4.54,253,Ramen Roma


## 4. Preprocesado del texto

Se procesa el campo `descripcion`:
- Eliminación de nulos
- Conversión a minúsculas


In [ ]:
df["descripcion"] = df["descripcion"].fillna("")
df["descripcion_procesada"] = df["descripcion"].str.lower()
df.head()


,id,nombre,tipo_cocina,localizacion,descripcion,puntuacion_media,num_valoraciones,contenido,descripcion_procesada
0,1,Fonda Do Sol,japonesa,Málaga,"Especializado en carnes a la brasa de larga maduración, resulta ideal para eventos de empresa. Dispone de menú del día con varias alternativas. Restaurante de cocina mexicana situado en un entorno...",4.06,330,Fonda Do Sol,"especializado en carnes a la brasa de larga maduración, resulta ideal para eventos de empresa. dispone de menú del día con varias alternativas. restaurante de cocina mexicana situado en un entorno..."
1,2,Burger Bella,gallega,Madrid,"La cocina es visible desde la sala principal. La mayoría de los ingredientes son de proximidad. Especializado en tacos mexicanos con salsas caseras, resulta ideal para comidas rápidas entre semana...",4.75,179,Burger Bella,"la cocina es visible desde la sala principal. la mayoría de los ingredientes son de proximidad. especializado en tacos mexicanos con salsas caseras, resulta ideal para comidas rápidas entre semana..."
2,3,Ramen Select,mexicana,Bilbao,La cocina es visible desde la sala principal. El equipo de cocina apuesta por sabores intensos. Restaurante de cocina italiana situado en un entorno animado y perfecto para grupos. Especializado e...,3.90,344,Ramen Select,la cocina es visible desde la sala principal. el equipo de cocina apuesta por sabores intensos. restaurante de cocina italiana situado en un entorno animado y perfecto para grupos. especializado e...
3,4,Taberna Rica,japonesa,A Coruña,"Especializado en carnes a la brasa de larga maduración, resulta ideal para reuniones informales. Restaurante de cocina vegana situado en un entorno tranquilo y silencioso. La música de fondo crea ...",4.37,492,Taberna Rica,"especializado en carnes a la brasa de larga maduración, resulta ideal para reuniones informales. restaurante de cocina vegana situado en un entorno tranquilo y silencioso. la música de fondo crea ..."
4,5,Burger Gourmet,vegana,Barcelona,Restaurante de cocina mexicana situado en un entorno elegante pero cercano. La presentación de los platos está muy cuidada. El servicio es cercano y cuida cada detalle. Especializado en ramen case...,4.11,96,Burger Gourmet,restaurante de cocina mexicana situado en un entorno elegante pero cercano. la presentación de los platos está muy cuidada. el servicio es cercano y cuida cada detalle. especializado en ramen case...


In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df["descripcion_procesada"])

print("TF-IDF shape:", tfidf_matrix.shape)


TF-IDF shape: (10000, 177)


## 6. Cálculo de similitud configurable

Se pueden emplear tres métricas:

- **Coseno**  
- **Euclidiana**  
- **Manhattan**  

La métrica euclidiana y manhattan se convierten a similitud en rango 0–1.


In [ ]:
def calcular_matriz_similitud(tfidf_matrix, metodo="coseno"):
    metodo = metodo.lower()

    if metodo == "coseno":
        return cosine_similarity(tfidf_matrix)

    elif metodo == "euclidiana":
        dist = euclidean_distances(tfidf_matrix)
        return 1 / (1 + dist)

    elif metodo == "manhattan":
        dist = manhattan_distances(tfidf_matrix)
        return 1 / (1 + dist)

    else:
        raise ValueError("Método no soportado: use coseno, euclidiana o manhattan.")


In [ ]:
metodo_similitud = "coseno"   # Cambiar aquí: "coseno", "euclidiana", "manhattan"

similarity_matrix = calcular_matriz_similitud(tfidf_matrix, metodo_similitud)

print("Matriz de similitud generada con:", metodo_similitud)
similarity_matrix[:4, :4]


Matriz de similitud generada con: coseno


array([[1.        , 0.19509428, 0.31083188, 0.41474463],
       [0.19509428, 1.        , 0.36927379, 0.17341787],
       [0.31083188, 0.36927379, 1.        , 0.20371367],
       [0.41474463, 0.17341787, 0.20371367, 1.        ]])

## 7. Fórmula personalizada de recomendación

Podemos combinar la similitud basada en texto con atributos como:

- Puntuación media
- Número de valoraciones
- Tipo de cocina
- Localización

Fórmula general:

score_final = w1 * similitud_texto
+ w2 * normalizar(puntuacion_media)
+ w3 * normalizar(num_valoraciones)
+ w4 * coincidencia_tipo_cocina
+ w5 * coincidencia_localizacion

In [ ]:
def recomendar(restaurante_id, df, similarity_matrix, n=5,
               w_sim=0.7, w_rating=0.2, w_votes=0.1,
               usar_tipo=True, usar_localizacion=True):

    if restaurante_id not in df["id"].values:
        raise ValueError("ID no encontrado.")

    idx = df.index[df["id"] == restaurante_id][0]

    # similitud base (texto)
    sim_base = similarity_matrix[idx]

    # normalización
    ratings_norm = (df["puntuacion_media"] - df["puntuacion_media"].min()) / \
                   (df["puntuacion_media"].max() - df["puntuacion_media"].min())

    votes_norm = (df["num_valoraciones"] - df["num_valoraciones"].min()) / \
                 (df["num_valoraciones"].max() - df["num_valoraciones"].min())

    # coincidencias categóricas
    base_tipo = df.loc[idx, "tipo_cocina"]
    base_loc = df.loc[idx, "localizacion"]

    match_tipo = (df["tipo_cocina"] == base_tipo).astype(int)
    match_loc = (df["localizacion"] == base_loc).astype(int)

    # fórmula final
    score = (
        w_sim * sim_base +
        w_rating * ratings_norm +
        w_votes * votes_norm +
        (usar_tipo * 0.05) * match_tipo +
        (usar_localizacion * 0.05) * match_loc
    )

    # Ordenar
    indices = np.argsort(score)[::-1]

    # Quitar el propio restaurante
    indices = [i for i in indices if i != idx][:n]

    resultados = df.iloc[indices].copy()
    resultados["score_final"] = score[indices]

    return resultados


In [ ]:
## 8. Búsqueda de restaurantes por nombre
def buscar(nombre):
    nombre = nombre.lower()
    mask = df["nombre"].str.lower().str.contains(nombre)
    return df[mask][["id","nombre","tipo_cocina","localizacion","descripcion"]].head(10)

# Guardamos el resultado de la búsqueda en una variable
resultados_busqueda = buscar("Cantina") #Cantina Osteria Trattoria Burger
print("Resultados de la búsqueda:")
print(resultados_busqueda)


Resultados de la búsqueda:
      id             nombre   tipo_cocina localizacion  \
30    31     Cantina Fusión     americana     A Coruña   
37    38  Cantina Atlántico         india         Lugo   
63    64    Cantina Lumière      japonesa     Santiago   
76    77   Cantina Delicias      japonesa       Málaga   
96    97      Cantina House  mediterránea         Vigo   
101  102      Cantina House         china      Sevilla   
108  109      Cantina House      japonesa     A Coruña   
130  131    Cantina Friends       gallega      Ourense   
131  132        Cantina Zen         india     A Coruña   
154  155       Cantina Rica        vegana     Zaragoza   

                                                                                                                                                                                                 descripcion  
30   Dispone de menú del día con varias alternativas. Restaurante de cocina fusión situado en un entorno acogedor y cálido. La 

In [ ]:
## 9. Lanzar el sistema de recomendación
# Elegimos un restaurante al azar
#rest_id = int(df.sample(1, random_state=1)["id"].iloc[0])
rest_id = int(resultados_busqueda["id"].iloc[0])

# Obtenemos el nombre del restaurante a partir del ID elegido
rest_nombre = df[df["id"] == rest_id]["nombre"].iloc[0]

print("Restaurante elegido [Busqueda] (ID):", rest_id)
print("Restaurante elegido [Busqueda] (Nombre):", rest_nombre)
print("---")

df[df["id"] == rest_id][["nombre","tipo_cocina","localizacion","descripcion"]]
recs = recomendar(
    restaurante_id=rest_id,
    df=df,
    similarity_matrix=similarity_matrix,
    n=5
)

recs

Restaurante elegido [Busqueda] (ID): 31
Restaurante elegido [Busqueda] (Nombre): Cantina Fusión
---


,id,nombre,tipo_cocina,localizacion,descripcion,puntuacion_media,num_valoraciones,contenido,descripcion_procesada,score_final
4754,4755,Veggie Gourmet,china,Santiago,"La relación calidad-precio es uno de sus puntos fuertes. Dispone de menú del día con varias alternativas. Especializado en tapas variadas para compartir, resulta ideal para eventos de empresa. Res...",4.99,279,Veggie Gourmet,"la relación calidad-precio es uno de sus puntos fuertes. dispone de menú del día con varias alternativas. especializado en tapas variadas para compartir, resulta ideal para eventos de empresa. res...",0.858822
1027,1028,Cantina Urbano,americana,Pontevedra,"Restaurante de cocina fusión situado en un entorno informal y desenfadado. Especializado en postres caseros inspirados en recetas locales, resulta ideal para celebraciones familiares. La relación ...",4.92,491,Cantina Urbano,"restaurante de cocina fusión situado en un entorno informal y desenfadado. especializado en postres caseros inspirados en recetas locales, resulta ideal para celebraciones familiares. la relación ...",0.855651
8437,8438,Tapas Castellana,healthy,Valencia,"Restaurante de cocina india situado en un entorno tranquilo y silencioso. Dispone de menú del día con varias alternativas. Especializado en tapas variadas para compartir, resulta ideal para comida...",4.90,313,Tapas Castellana,"restaurante de cocina india situado en un entorno tranquilo y silencioso. dispone de menú del día con varias alternativas. especializado en tapas variadas para compartir, resulta ideal para comida...",0.800204
3302,3303,Pizzería Zen,italiana,Madrid,"Restaurante de cocina gallega situado en un entorno acogedor y cálido. Especializado en tapas variadas para compartir, resulta ideal para celebraciones familiares. El servicio es cercano y cuida c...",4.80,492,Pizzería Zen,"restaurante de cocina gallega situado en un entorno acogedor y cálido. especializado en tapas variadas para compartir, resulta ideal para celebraciones familiares. el servicio es cercano y cuida c...",0.779717
6908,6909,Ramen Friends,vegana,Sevilla,La relación calidad-precio es uno de sus puntos fuertes. Restaurante de cocina italiana situado en un entorno acogedor y cálido. Dispone de menú del día con varias alternativas. Especializado en t...,4.37,448,Ramen Friends,la relación calidad-precio es uno de sus puntos fuertes. restaurante de cocina italiana situado en un entorno acogedor y cálido. dispone de menú del día con varias alternativas. especializado en t...,0.753382


## 10. Conclusiones

Este notebook demuestra:

- Carga de un dataset grande (10.000 restaurantes)
- Preprocesado de descripciones
- Construcción de TF-IDF
- Sistema de similitud configurable (coseno, euclídea, manhattan)
- Sistema de recomendación refinable mediante fórmula personalizada
- Búsqueda y recomendación funcional

Este módulo forma la base del **motor de recomendaciones** del proyecto DELICHOICE.


## Recursos

### Dataset de restaurantes

El sistema utiliza como recurso principal el siguiente conjunto de datos:

- **Nombre:** `restaurantes_delichoice_10000_unique_contenido.csv`
- **Descripción:** Dataset con 10.000 restaurantes únicos que incluye información descriptiva y de valoración.
- **Campos principales:**  
  `id`, `nombre`, `tipo_cocina`, `localizacion`, `descripcion`, `puntuacion_media`, `num_valoraciones`
- **Uso dentro de la aplicación:**
  - En el **sistema de recomendación**, se emplea el campo `descripcion` (junto con otros campos textuales) para generar representaciones TF-IDF.
  - Los campos `puntuacion_media` y `num_valoraciones` se utilizan como información adicional para mostrar calidad y popularidad.
- **Formato:** CSV (Comma-Separated Values)
- **Ubicación:** Mismo directorio que el notebook o ruta configurada en la celda de carga de datos.

> Nota: este dataset es necesario para ejecutar correctamente el sistema de recomendación y está adjuntado en el zip de la entrega.



## Bibliografía

- ABP. (s.f.). *Tutorial: Recomendador basado en contenido*.  
  GitHub. Recuperado el 9 de enero de 2026, de  
  https://github.com/adrseara/abp_notebooks/blob/master/Tutorial_Recomendador_basado_en_contenido.ipynb
